In [ ]:
# GenAI RAG Agent - Final Project


In [18]:
## 1. Setup (your pip installs, API key, imports)

!pip install groq sentence-transformers wikipedia scikit-learn

from groq import Groq
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import wikipedia
import json

client = Groq(api_key="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")


In [19]:
## 2. Document Loading & Chunking (your wikipedia fetch + chunk_text function)

article = wikipedia.page("Artificial intelligence")
document_text = article.content

def chunk_text(text, chunk_size=1000):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)
    return chunks

chunks = chunk_text(document_text)

In [20]:
## 3. Embeddings & Retrieval (your embedder + retrieve_relevant_chunk, your BEST version — top_n=4)

embedder = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embedder.encode(chunks)

def retrieve_relevant_chunk(question, top_n=4):  # top_n=4 was our winning version
    question_embedding = embedder.encode([question])
    similarities = cosine_similarity(question_embedding, chunk_embeddings)[0]
    top_indices = similarities.argsort()[-top_n:][::-1]
    relevant_chunks = [chunks[i] for i in top_indices]
    return relevant_chunks

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [21]:
## 4. RAG Chat Function (your final grounded rag_chat / get_rag_answer)

def get_rag_answer(question, top_n=4):
    relevant_chunks = retrieve_relevant_chunk(question, top_n=top_n)
    context = "\n\n".join(relevant_chunks)

    system_prompt = f"""You are a helpful assistant. Answer the user's question using ONLY the context below.
If the answer isn't in the context, say "I don't know based on the provided document."

Context:
{context}"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content

# Version that also returns token cost (used in Day 20)
def get_rag_answer_with_cost(question, top_n=4):
    relevant_chunks = retrieve_relevant_chunk(question, top_n=top_n)
    context = "\n\n".join(relevant_chunks)

    system_prompt = f"""You are a helpful assistant. Answer the user's question using ONLY the context below.
If the answer isn't in the context, say "I don't know based on the provided document."

Context:
{context}"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question}
        ]
    )
    return response.choices[0].message.content, response.usage.total_tokens

In [22]:
## 5. Evaluation (your eval_set, judge_answer, and the final comparison table)

eval_set = [
    {"question": "In what year was artificial intelligence officially founded as an academic discipline?", "expected": "1956"},
    {"question": "Who originally coined the term \"artificial intelligence\"?", "expected": "John McCarthy"},
    {"question": "What specific term is used to describe historical periods where AI research experienced a loss of funding and a wave of institutional disappointment?", "expected": "AI winters"},
    {"question": "Name three high-profile applications of AI listed in the article's introduction.", "expected": "Advanced web search engines, chatbots, virtual assistants, autonomous vehicles, or strategy game engines (like chess and Go)"},
    {"question": "What is the term for a formal representation of knowledge as a set of concepts within a domain and the relationships between them?", "expected": "An ontology"},
    {"question": "Besides engineering, mathematics, and computer science, what are some of the other interdisciplinary fields that AI draws upon?", "expected": "Psychology, linguistics, philosophy, and neuroscience"},
    {"question": "Summarize the traditional goals of AI research as outlined in the article.", "expected": "The core traditional goals include learning, reasoning, knowledge representation, planning, natural language processing, perception, and support for robotics."},
    {"question": "How does an AI agent calculate and use \"expected utility\" to make choices?", "expected": "For each possible action, the agent calculates the utility of all potential outcomes, weights those outcomes by the mathematical probability that they will occur, and then selects the specific action that maximizes this overall value."},
    {"question": "What is the difference between \"machine perception\" and \"computer vision\" based on the text?", "expected": "Machine perception is the broad, overarching capability to use input from any type of sensor (including microphones, lidar, sonar, and radar) to deduce aspects of the physical world. Computer vision is a specific subfield under that umbrella dedicated strictly to analyzing visual input."},
    {"question": "What components make up a Markov decision process according to the article's framework?", "expected": "A Markov decision process relies on a transition model (which describes the probability that a particular action will change the state in a specific way) and a reward function (which supplies the utility of each state and the cost of each action)."},
    {"question": "How does the article explain the real-world utility of formal knowledge representations?", "expected": "They allow AI systems to answer questions intelligently and deduce real-world facts. They are actively applied in clinical decision support, scene interpretation, content-based indexing/retrieval, and automated knowledge discovery in massive databases."},
    {"question": "How does the article characterize the rapid evolution of generative AI since the 2020s?", "expected": "It notes that generative AI has become widely accessible to the public, allowing users to generate high-quality images, audio, and videos purely from basic text prompts."},
    {"question": "What exact percentage of global carbon emissions was directly caused by AI data centers in the year 2025?", "expected": "I don't know. The article notes that power needs and environmental impacts are recognized risks of AI, but it does not provide any specific statistical breakdowns or concrete percentage data for global emissions in 2025."},
    {"question": "How many lines of code did Allen Newell and Herbert A. Simon write to build the initial version of the Logic Theorist program presented at the Dartmouth Workshop?", "expected": "I don't know. The article mentions that the Logic Theorist was presented at the 1956 workshop as the first AI program, but it contains absolutely no information regarding its source code length, language implementation, or code lines."},
    {"question": "Which specific textbook is currently ranked as the absolute most widely used resource for teaching AI in universities worldwide?", "expected": "I don't know. While the text discusses AI as an established academic and scientific research discipline, it does not track, reference, or rank textbook sales, authors, or university syllabi."},
    {"question": "What proprietary algorithms did tech companies use to transition their web search engines from keyword indexing to generative semantic matching?", "expected": "I don't know. The article states that advanced web search engines are a high-profile application of AI, but it does not disclose or detail the internal corporate codebases, proprietary engineering designs, or specific mathematical transitions used by commercial engines."},
    {"question": "What was the exact financial budget surplus or deficit of the 1956 Dartmouth Workshop after attendee registration fees were processed?", "expected": "I don't know. The article outlines the historical significance of the Dartmouth Workshop as the birthplace of AI, but it completely omits any logistical financial records, costs, or registration details of the event."},
    {"question": "Exactly how many parameters are configured within the primary neural network models that handle autonomous vehicle navigation today?", "expected": "I don't know. The text explicitly mentions autonomous vehicles as an application of machine perception and AI, but it never details the technical specifications, architecture layers, or parameter counts of actual modern driving models."}
]


def judge_answer(question, expected, actual):
    judge_prompt = f"""You are grading whether an AI's answer is correct.

Question: {question}
Expected answer: {expected}
AI's actual answer: {actual}

Does the AI's actual answer correctly capture the expected answer, even if worded differently?
Respond with ONLY "PASS" or "FAIL", followed by a one-sentence reason.
Format exactly like this:
PASS: reason here
or
FAIL: reason here"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": judge_prompt}]
    )
    return response.choices[0].message.content

# Run the full eval
results = []
total_tokens_used = 0

for item in eval_set:
    question = item["question"]
    expected = item["expected"]
    actual, tokens = get_rag_answer_with_cost(question)
    verdict = judge_answer(question, expected, actual)

    total_tokens_used += tokens
    results.append({
        "question": question,
        "expected": expected,
        "actual": actual,
        "verdict": verdict,
        "tokens": tokens
    })

passes = sum(1 for r in results if r["verdict"].startswith("PASS"))
total = len(results)
score = passes / total

print(f"Score: {passes}/{total} ({score*100:.1f}%)")
print(f"Average tokens per question: {total_tokens_used / total:.0f}")


Score: 15/18 (83.3%)
Average tokens per question: 894


In [23]:
## =========================================
## 6. AGENT TOOL USE
## =========================================

def get_word_count(text):
    return len(text.split())

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_word_count",
            "description": "Counts the number of words in a given piece of text.",
            "parameters": {
                "type": "object",
                "properties": {
                    "text": {
                        "type": "string",
                        "description": "The text to count words in"
                    }
                },
                "required": ["text"]
            }
        }
    }
]

def safe_execute_tool(tool_call):
    try:
        function_args = json.loads(tool_call.function.arguments)
        if tool_call.function.name == "get_word_count":
            result = get_word_count(function_args["text"])
            return str(result)
        else:
            return f"Error: unknown function '{tool_call.function.name}'"
    except json.JSONDecodeError:
        return "Error: the tool arguments were not valid JSON"
    except KeyError as e:
        return f"Error: missing required argument {e}"
    except Exception as e:
        return f"Error: something went wrong — {str(e)}"

# Full agent loop
def run_agent(user_question):
    messages = [{"role": "user", "content": user_question}]

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        tools=tools
    )

    response_message = response.choices[0].message
    messages.append(response_message)

    if response_message.tool_calls:
        tool_call = response_message.tool_calls[0]
        result = safe_execute_tool(tool_call)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })

        final_response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=messages
        )
        return final_response.choices[0].message.content
    else:
        return response_message.content

# Test it
print(run_agent("How many words are in the sentence: The quick brown fox jumps over the lazy dog"))

There are 9 words in the sentence: 

1. The
2. quick
3. brown
4. fox
5. jumps
6. over
7. the
8. lazy
9. dog
